In [1]:
!pip install pandas geopandas shapely scikit-learn geopy matplotlib seaborn

In [2]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
pickup_df = pd.read_csv('/content/drive/MyDrive/data/cleaned_pickup_data.csv')
delivery_df = pd.read_csv('/content/drive/MyDrive/data/cleaned_delivery_data_edit.csv')
print(pickup_df.head())
print(delivery_df.head())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   order_id  region_id   city  courier_id          accept_time  \
0    483671          3  City1        1518  1900-08-14 07:57:00   
1   1746131          3  City1        4706  1900-10-09 07:46:00   
2   2301722          3  City1        4706  1900-10-09 13:57:00   
3   3788723          3  City1        4706  1900-05-19 08:13:00   
4    713435          3  City1        4706  1900-05-22 08:16:00   

     time_window_start      time_window_end        lng       lat  aoi_id  ...  \
0  1900-08-14 09:00:00  1900-08-14 11:00:00  106.46877  29.47204     218  ...   
1  1900-10-09 09:00:00  1900-10-09 11:00:00  106.46872  29.47200     218  ...   
2  1900-10-09 13:57:00  1900-10-09 15:57:00  106.46869  29.47191     218  ...   
3  1900-05-19 11:00:00  1900-05-19 13:00:00  106.46878  29.47208     218  ...   
4  1900-05-22 09:00:00  1900-05-22 11:00:00  106.46813  29.47228     

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:

print(pickup_df.isnull().sum())
print(delivery_df.isnull().sum())

print(pickup_df.duplicated().sum())
print(delivery_df.duplicated().sum())

print(pickup_df.dtypes)
print(delivery_df.dtypes)

order_id                0
region_id               0
city                    0
courier_id              0
accept_time             0
time_window_start       0
time_window_end         0
lng                     0
lat                     0
aoi_id                  0
aoi_type                0
pickup_time             0
pickup_gps_time         0
pickup_gps_lng          0
pickup_gps_lat          0
accept_gps_time         0
accept_gps_lng          0
accept_gps_lat          0
ds                      0
time_window_duration    0
task_duration           0
distance                0
dtype: int64
order_id                          0
region_id                         0
city_name                         0
courier_id                        0
lng                               0
lat                               0
aoi_id                            0
aoi_type                          0
accept_time                       0
accept_gps_time                   0
accept_gps_lng                    0
accept_gps_lat     

In [5]:
merged_df = pd.merge(pickup_df, delivery_df, on='order_id', how='inner')

columns_to_keep = [
    'order_id', 'pickup_time', 'pickup_gps_lng', 'pickup_gps_lat',
    'city', 'region_id', 'aoi_id', 'aoi_type', 'courier_id',
    'delivery_time', 'delivery_gps_lng', 'delivery_gps_lat'
]
print(merged_df.head())

   order_id  region_id_x city_x  courier_id_x        accept_time_x  \
0    483671            3  City1          1518  1900-08-14 07:57:00   
1   1746131            3  City1          4706  1900-10-09 07:46:00   
2   2301722            3  City1          4706  1900-10-09 13:57:00   
3    713435            3  City1          4706  1900-05-22 08:16:00   
4   2718201            3  City1          4706  1900-05-19 07:43:00   

     time_window_start      time_window_end      lng_x     lat_x  aoi_id_x  \
0  1900-08-14 09:00:00  1900-08-14 11:00:00  106.46877  29.47204       218   
1  1900-10-09 09:00:00  1900-10-09 11:00:00  106.46872  29.47200       218   
2  1900-10-09 13:57:00  1900-10-09 15:57:00  106.46869  29.47191       218   
3  1900-05-22 09:00:00  1900-05-22 11:00:00  106.46813  29.47228       218   
4  1900-05-19 09:00:00  1900-05-19 11:00:00  106.46869  29.47206       218   

   ...        delivery_time delivery_gps_time delivery_gps_lng  \
0  ...  2024-08-08 17:37:00    8/8/2024 17:3

In [6]:
merged_df['pickup_time'] = pd.to_datetime(merged_df['pickup_time'])
merged_df['delivery_time'] = pd.to_datetime(merged_df['delivery_time'])

merged_df['ETA'] = (merged_df['delivery_time'] - merged_df['pickup_time']).dt.total_seconds()
print(merged_df.head())

   order_id  region_id_x city_x  courier_id_x        accept_time_x  \
0    483671            3  City1          1518  1900-08-14 07:57:00   
1   1746131            3  City1          4706  1900-10-09 07:46:00   
2   2301722            3  City1          4706  1900-10-09 13:57:00   
3    713435            3  City1          4706  1900-05-22 08:16:00   
4   2718201            3  City1          4706  1900-05-19 07:43:00   

     time_window_start      time_window_end      lng_x     lat_x  aoi_id_x  \
0  1900-08-14 09:00:00  1900-08-14 11:00:00  106.46877  29.47204       218   
1  1900-10-09 09:00:00  1900-10-09 11:00:00  106.46872  29.47200       218   
2  1900-10-09 13:57:00  1900-10-09 15:57:00  106.46869  29.47191       218   
3  1900-05-22 09:00:00  1900-05-22 11:00:00  106.46813  29.47228       218   
4  1900-05-19 09:00:00  1900-05-19 11:00:00  106.46869  29.47206       218   

   ...  delivery_gps_time delivery_gps_lng delivery_gps_lat  ds_y  city_y  \
0  ...     8/8/2024 17:37        

In [7]:
print(merged_df.isnull().sum())
merged_df.dropna(subset=['pickup_time', 'delivery_time', 'pickup_gps_lng', 'pickup_gps_lat', 'delivery_gps_lng', 'delivery_gps_lat'], inplace=True)
print(merged_df.isnull().sum())

order_id                          0
region_id_x                       0
city_x                            0
courier_id_x                      0
accept_time_x                     0
time_window_start                 0
time_window_end                   0
lng_x                             0
lat_x                             0
aoi_id_x                          0
aoi_type_x                        0
pickup_time                       0
pickup_gps_time                   0
pickup_gps_lng                    0
pickup_gps_lat                    0
accept_gps_time_x                 0
accept_gps_lng_x                  0
accept_gps_lat_x                  0
ds_x                              0
time_window_duration              0
task_duration                     0
distance_x                        0
region_id_y                       0
city_name                         0
courier_id_y                      0
lng_y                             0
lat_y                             0
aoi_id_y                    

In [8]:
merged_df.to_csv('/content/drive/My Drive/merged_dataset.csv', index=False)

In [9]:
from IPython import get_ipython
from IPython.display import display
!pip install pandas geopandas shapely scikit-learn geopy matplotlib seaborn
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

roads_df = pd.read_csv('/content/drive/MyDrive/data/roads.csv', sep='\t', on_bad_lines='skip')

from shapely.geometry import Point
pickup_points = gpd.GeoDataFrame(
    merged_df,
    geometry=gpd.points_from_xy(merged_df['pickup_gps_lng'], merged_df['pickup_gps_lat']),
    crs="EPSG:4326"
)
delivery_points = gpd.GeoDataFrame(
    merged_df,
    geometry=gpd.points_from_xy(merged_df['delivery_gps_lng'], merged_df['delivery_gps_lat']),
    crs="EPSG:4326"
)
print(pickup_points.head())


   order_id  region_id_x city_x  courier_id_x        accept_time_x  \
0    483671            3  City1          1518  1900-08-14 07:57:00   
1   1746131            3  City1          4706  1900-10-09 07:46:00   
2   2301722            3  City1          4706  1900-10-09 13:57:00   
3    713435            3  City1          4706  1900-05-22 08:16:00   
4   2718201            3  City1          4706  1900-05-19 07:43:00   

     time_window_start      time_window_end      lng_x     lat_x  aoi_id_x  \
0  1900-08-14 09:00:00  1900-08-14 11:00:00  106.46877  29.47204       218   
1  1900-10-09 09:00:00  1900-10-09 11:00:00  106.46872  29.47200       218   
2  1900-10-09 13:57:00  1900-10-09 15:57:00  106.46869  29.47191       218   
3  1900-05-22 09:00:00  1900-05-22 11:00:00  106.46813  29.47228       218   
4  1900-05-19 09:00:00  1900-05-19 11:00:00  106.46869  29.47206       218   

   ...  delivery_gps_lng delivery_gps_lat  ds_y  city_y  delivery_duration  \
0  ...         121.53092        

In [ ]:
!pip install shapely
from shapely.wkt import loads
roads_df['geometry'] = roads_df['geometry'].apply(loads)

roads_gdf = gpd.GeoDataFrame(roads_df, geometry='geometry', crs="EPSG:4326")
pickup_with_roads = gpd.sjoin_nearest(pickup_points, roads_gdf, how='left', distance_col='distance_to_road')

pickup_with_roads = pickup_with_roads[['package_id', 'maxspeed', 'fclass', 'oneway', 'bridge', 'tunnel', 'distance_to_road']]


print(pickup_with_roads.head())
roads_gdf = gpd.GeoDataFrame(roads_df, geometry='geometry', crs="EPSG:4326")
pickup_with_roads = gpd.sjoin_nearest(pickup_points, roads_df, how='left', distance_col='distance_to_road')

pickup_with_roads = pickup_with_roads[['package_id', 'maxspeed', 'fclass', 'oneway', 'bridge', 'tunnel', 'distance_to_road']]

print(pickup_with_roads.head())



/usr/local/lib/python3.11/dist-packages/geopandas/array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
